# 06 - Model Evaluation & Comparison

Comprehensive evaluation of trained models including residual analysis, prediction intervals, learning curves, and model comparison report generation.

In [ ]:
# Setup
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.data.preprocessing import load_processed_data
from src.data.feature_engineering import full_feature_engineering_pipeline
from src.models.train import load_training_results
from src.models.evaluate import (
    evaluate_model_detailed,
    compare_models,
    plot_residuals,
    plot_model_comparison,
    plot_feature_importance_comparison,
    plot_learning_curves,
    generate_evaluation_report
)
from src.config import DATA_PROCESSED, MODELS_DIR, REPORTS_DIR

In [ ]:
# Load Data and Models
X_train, X_test, y_train, y_test, scaler = load_processed_data()

# Load engineered test data for evaluation
test_engineered = pd.read_csv(DATA_PROCESSED / 'kc_house_test_engineered.csv')

# Prepare features and target from engineered data
drop_cols = ['id', 'date', 'price', 'price_per_sqft', 'age_band']
drop_cols = [c for c in drop_cols if c in test_engineered.columns]

X_test_eng = test_engineered.drop(columns=drop_cols)
y_test_eng = test_engineered['price']

# Handle categorical columns
cat_cols = X_test_eng.select_dtypes(include=['object']).columns
if len(cat_cols) > 0:
    X_test_eng = pd.get_dummies(X_test_eng, columns=cat_cols, drop_first=True)

# Load feature names from training
X_train_eng = pd.read_csv(DATA_PROCESSED / 'kc_house_engineered.csv')
X_train_eng = X_train_eng.drop(columns=drop_cols)
cat_cols = X_train_eng.select_dtypes(include=['object']).columns
if len(cat_cols) > 0:
    X_train_eng = pd.get_dummies(X_train_eng, columns=cat_cols, drop_first=True)

# Align test with training features
X_test_eng = X_test_eng.reindex(columns=X_train_eng.columns, fill_value=0)

feature_names = X_train_eng.columns.tolist()

results = load_training_results()

print(f"Loaded {len(results)} models")
print(f"Test set: {X_test_eng.shape[0]} samples")
print(f"Features: {len(feature_names)}")

In [ ]:
# 1. Model Comparison on Test Set
print("=== 1. Model Comparison ===")
from src.config import MODELS_DIR
comparison_df = compare_models(results, X_test_eng, y_test_eng, models_dir=MODELS_DIR)
print(comparison_df[['model', 'MAE', 'RMSE', 'R2', 'MAPE', 'cv_rmse_mean', 'cv_rmse_std']].to_string(index=False))

In [ ]:
# 2. Detailed Evaluation
print("=== 2. Detailed Evaluation ===")

import joblib
from src.config import MODELS_DIR

detailed_results = {}
for name, result in results.items():
    if "error" in result and result["error"] is not None:
        continue
    
    # Load model from disk
    model_path = MODELS_DIR / f"{name}.pkl"
    if not model_path.exists():
        continue
    model = joblib.load(model_path)
    
    detailed_results[name] = evaluate_model_detailed(model, X_test, y_test, name)
    
    dr = detailed_results[name]
    print(f"\n{name}:")
    print(f"  MAE: {dr['metrics']['MAE']:,.0f}")
    print(f"  RMSE: {dr['metrics']['RMSE']:,.0f}")
    print(f"  R²: {dr['metrics']['R2']:.4f}")
    print(f"  Residual Mean: {dr['residual_stats']['mean']:,.0f}")
    print(f"  Residual Std: {dr['residual_stats']['std']:,.0f}")
    print(f"  Shapiro-Wilk p: {dr['normality_tests']['shapiro_p']:.4f}")
    print(f"  95% PI: ±${dr['prediction_intervals']['95%']:,.0f}")

In [ ]:
# 3. Residual Plots
print("=== 3. Residual Analysis Plots ===")

import joblib
from src.config import MODELS_DIR

# Load models for plotting
models_for_plot = {}
for name in comparison_df.head(4)['model']:
    model_path = MODELS_DIR / f"{name}.pkl"
    if model_path.exists():
        models_for_plot[name] = joblib.load(model_path)

# Create a results dict with models for the plotting function
results_with_models = {}
for name, model in models_for_plot.items():
    results_with_models[name] = {"model": model}

fig = plot_residuals(results_with_models, X_test_eng, y_test_eng, top_n=len(models_for_plot), save_path=REPORTS_DIR / 'figures' / 'residuals.png')
plt.show()

In [ ]:
# 4. Model Comparison Bar Charts
print("=== 4. Model Comparison Charts ===")
fig = plot_model_comparison(comparison_df, save_path=REPORTS_DIR / 'figures' / 'model_comparison.png')
plt.show()

In [ ]:
# 5. Feature Importance Comparison
print("=== 5. Feature Importance Comparison ===")

import joblib
from src.config import MODELS_DIR

results_with_models = {}
for name in comparison_df['model']:
    model_path = MODELS_DIR / f"{name}.pkl"
    if model_path.exists():
        results_with_models[name] = {"model": joblib.load(model_path)}

fig = plot_feature_importance_comparison(results_with_models, feature_names, top_k=15, save_path=REPORTS_DIR / 'figures' / 'feature_importance.png')
if fig:
    plt.show()

In [ ]:
# 6. Learning Curves for Best Model
print("=== 6. Learning Curves ===")
best_name = comparison_df.iloc[0]['model']
best_model = results[best_name]['model']

fig = plot_learning_curves(best_model, X_train, y_train, save_path=REPORTS_DIR / 'figures' / f'learning_curves_{best_name}.png')
plt.show()

In [ ]:
# 7. Generate Full Evaluation Report
print("=== 7. Generating Evaluation Report ===")
eval_results = generate_evaluation_report(results, X_test, y_test, feature_names)

print(f"\nBest model: {eval_results['best_model']}")
print(f"Report saved to: {REPORTS_DIR / 'model_comparison.md'}")

In [ ]:
# 8. Error Analysis by Price Range
print("=== 8. Error Analysis by Price Range ===")

for name in comparison_df.head(3)['model']:
    if name in detailed_results:
        dr = detailed_results[name]
        print(f"\n{name} - Error by Price Bin:")
        for bin_name, stats in dr['error_by_price_bin'].items():
            if isinstance(stats, dict) and 'residual' in stats:
                print(f"  {bin_name}: Mean Residual=${stats['residual']['mean']:,.0f}, Std=${stats['residual']['std']:,.0f}, Count={stats['actual']['count']}")

In [ ]:
# 9. Prediction Intervals Summary
print("=== 9. Prediction Intervals ===")

for name in comparison_df.head(3)['model']:
    if name in detailed_results:
        dr = detailed_results[name]
        pi = dr['prediction_intervals']
        print(f"{name}: 95% PI = ±${pi['95%']:,.0f}, 80% PI = ±${pi['80%']:,.0f}")

In [ ]:
# 10. Residual Diagnostics Summary
print("=== 10. Residual Diagnostics Summary ===")

diag_summary = []
for name in comparison_df['model']:
    if name in detailed_results:
        dr = detailed_results[name]
        diag_summary.append({
            'model': name,
            'residual_mean': dr['residual_stats']['mean'],
            'residual_std': dr['residual_stats']['std'],
            'skew': dr['residual_stats']['skew'],
            'shapiro_p': dr['normality_tests']['shapiro_p'],
            'normal_shapiro': dr['normality_tests']['is_normal_shapiro'],
            'bp_p': dr['heteroscedasticity'].get('p_value', np.nan),
            'heteroscedastic': dr['heteroscedasticity'].get('heteroscedastic', np.nan),
            'pi_95': dr['prediction_intervals']['95%']
        })

diag_df = pd.DataFrame(diag_summary)
print(diag_df.to_string(index=False))

In [ ]:
# 11. Save Valuation Output
print("=== 11. Saving Valuation Output ===")

# Generate predictions for test set with best model
best_model = results[eval_results['best_model']]['model']
y_pred = best_model.predict(X_test)

valuation_df = pd.DataFrame({
    'actual_price': y_test.values,
    'predicted_price': y_pred,
    'residual': y_test.values - y_pred,
    'abs_error': np.abs(y_test.values - y_pred),
    'pct_error': np.abs((y_test.values - y_pred) / y_test.values) * 100
})

# Add feature info for context
for col in ['sqft_living', 'bedrooms', 'bathrooms', 'grade', 'zipcode']:
    if col in X_test.columns:
        valuation_df[col] = X_test[col].values

valuation_path = REPORTS_DIR / 'valuation_output.csv'
valuation_df.to_csv(valuation_path, index=False)
print(f"Valuation output saved to: {valuation_path}")

# Summary stats
print(f"\nValuation Summary:")
print(f"  MAE: ${valuation_df['abs_error'].mean():,.0f}")
print(f"  Median Abs Error: ${valuation_df['abs_error'].median():,.0f}")
print(f"  MAPE: {valuation_df['pct_error'].mean():.1f}%")
print(f"  Within 10%: {(valuation_df['pct_error'] < 10).mean()*100:.1f}%")
print(f"  Within 20%: {(valuation_df['pct_error'] < 20).mean()*100:.1f}%")